In [2]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
BREVO_KEY = os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")
llm_gpt = ChatCerebras(
    model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"),
)
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [43]:
class User(BaseModel):
    id: int
    name: str
    email: str
    minutes_listened: int
    country: str
    joined_at: str


class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str
    reason: str


class CampaignResult(BaseModel):
    total_users: int
    emails_sent: int
    failed: int


In [ ]:
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["soulsync"]

users_collection = db["users"]

print(users_collection)


Collection(Database(MongoClient(host=['ac-jaaey03-shard-00-00.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-01.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-02.k7w05ba.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='SoulSync', authsource='admin', replicaset='atlas-99cwym-shard-0', tls=True), 'soulsync'), 'users')


In [ ]:
print(db.list_collection_names())

NameError: name 'users' is not defined

In [44]:
users = [
    User(
        id=1,
        name="Alex",
        email="alex@example.com",
        minutes_listened=8420,
        country="USA",
        joined_at="2024-01-15"
    ),
    User(
        id=2,
        name="Emma",
        email="emma@example.com",
        minutes_listened=7560,
        country="UK",
        joined_at="2024-03-10"
    ),
    User(
        id=3,
        name="John",
        email="john@example.com",
        minutes_listened=4320,
        country="Canada",
        joined_at="2024-05-21"
    ),
    User(
        id=4,
        name="Sarah",
        email="sarah@example.com",
        minutes_listened=9680,
        country="Australia",
        joined_at="2023-11-08"
    ),
    User(
        id=5,
        name="Mike",
        email="mike@example.com",
        minutes_listened=3890,
        country="Germany",
        joined_at="2024-07-12"
    ),
    User(
        id=6,
        name="Sophia",
        email="sophia@example.com",
        minutes_listened=6150,
        country="France",
        joined_at="2024-02-18"
    ),
    User(
        id=7,
        name="David",
        email="david@example.com",
        minutes_listened=10540,
        country="India",
        joined_at="2023-12-02"
    ),
    User(
        id=8,
        name="Olivia",
        email="olivia@example.com",
        minutes_listened=4920,
        country="Italy",
        joined_at="2024-06-30"
    ),
    User(
        id=9,
        name="Daniel",
        email="daniel@example.com",
        minutes_listened=2780,
        country="Brazil",
        joined_at="2025-01-05"
    ),
    User(
        id=10,
        name="Emily",
        email="emily@example.com",
        minutes_listened=8340,
        country="Japan",
        joined_at="2024-04-11"
    ),
    User(
        id=11,
        name="James",
        email="james@example.com",
        minutes_listened=7290,
        country="Singapore",
        joined_at="2024-01-28"
    ),
    User(
        id=12,
        name="Charlotte",
        email="charlotte@example.com",
        minutes_listened=5430,
        country="Spain",
        joined_at="2024-08-14"
    ),
    User(
        id=13,
        name="Ethan",
        email="ethan@example.com",
        minutes_listened=11980,
        country="USA",
        joined_at="2023-10-22"
    ),
    User(
        id=14,
        name="Mia",
        email="mia@example.com",
        minutes_listened=3610,
        country="India",
        joined_at="2025-02-03"
    ),
    User(
        id=15,
        name="Noah",
        email="noah@example.com",
        minutes_listened=6870,
        country="South Africa",
        joined_at="2024-09-19"
    ),
    User(
        id=16,
        name="Loki",
        email="itslokeshx@gmail.com",
        minutes_listened=451000,
        country="India",
        joined_at="2026-03-14"
    ),
    User(
        id=17,
        name="Liam",
        email="liam@example.com",
        minutes_listened=9730,
        country="Ireland",
        joined_at="2023-09-17"
    ),
    User(
        id=18,
        name="Isabella",
        email="isabella@example.com",
        minutes_listened=5820,
        country="Netherlands",
        joined_at="2024-12-24"
    ),
    User(
        id=19,
        name="Benjamin",
        email="benjamin@example.com",
        minutes_listened=2410,
        country="Sweden",
        joined_at="2025-03-15"
    ),
    User(
        id=20,
        name="Grace",
        email="grace@example.com",
        minutes_listened=8950,
        country="New Zealand",
        joined_at="2024-05-05"
    ),
]

In [45]:
@tool
def GetTopUsers(limit:int):
     """
     Return the top users ranked by listening minutes.
     Args:
        limit: Number of top users to return.
     """
     sorted_users = sorted(
        users,
        key=lambda user: user.minutes_listened,
        reverse=True
     )
     return sorted_users[:limit]
    

In [63]:
@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str,
) -> str:
    """Send an email using Brevo."""

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": "SoulSync", "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)
        print(sender, receiver, subject)
        return f"Mail sent successfully to {receiver}"
    except ApiException as e:
        print(" Error:", e)
        return f"Failed to send email: {e}"


In [67]:
draft_llm = llm_gpt.with_structured_output(EmailDraft)
top_users = GetTopUsers.invoke(
    {
        "limit": 5
    }
)
drafts = []

for user in top_users:

    draft = draft_llm.invoke(f"""
    You are LOOKOUT.

    Write a professional appreciation email.

    Recipient: {user.name}
        Email: {user.email}

    Reason:
    The user is one of our top listeners this month with
    {user.minutes_listened} listening minutes.

    Write a warm thank-you email.
    """)

    drafts.append(draft)

In [68]:
drafts

[EmailDraft(recipient='itslokeshx@gmail.com', subject='Thank You for Being a Top Listener!', body='Dear Loki,\n\nI hope you’re doing well. I’m reaching out to extend our heartfelt gratitude for your incredible support this month. With an astounding 451,000 listening minutes, you’ve been one of our top listeners, and your dedication truly stands out.\n\nYour enthusiasm for our content not only inspires our team but also helps shape the future of our platform. It’s listeners like you who drive us to continuously improve and bring fresh, engaging experiences to our community.\n\nThank you for your continued loyalty and for being such an integral part of our journey. We look forward to bringing you even more content that you’ll love in the coming months.\n\nWarm regards,\n[Your Name]\n[Your Position]\nLOOKOUT Team', reason='The user is one of our top listeners this month with 451000 listening minutes.'),
 EmailDraft(recipient='Ethan', subject='Thank You for Being One of Our Top Listeners!'

In [57]:
for draft in drafts:
    print("=" * 60)
    print(f"Recipient : {draft.recipient}")
    print(f"Subject   : {draft.subject}")
    print(f"Reason    : {draft.reason}")
    print("-" * 60)
    print(draft.body)
    print("=" * 60)

Recipient : itslokeshx@gmail.com
Subject   : A Heartfelt Thank You from LOOKOUT
Reason    : Top listener of the month with 451,000 listening minutes
------------------------------------------------------------
Dear Loki,

We are thrilled to express our sincerest gratitude for being one of our top listeners this month with an impressive 451,000 listening minutes! Your dedication to our content is truly valued and appreciated.

Your loyalty and commitment inspire us to continue delivering high-quality experiences for our users. We are grateful for your engagement and trust in our platform.

Thank you for being an integral part of our LOOKOUT community. We look forward to continuing to provide you with the best content possible.

Warm regards,
LOOKOUT Team
Recipient : ethan@example.com
Subject   : A Big Thank You from Lookout!
Reason    : Appreciation for being one of our top listeners this month with 11,980 listening minutes.
------------------------------------------------------------
D

In [69]:
agent = create_agent(
    model=llm_gpt,
    tools=[GetTopUsers, sendMail],
    system_prompt="""
You are LOOKOUT.

You have two tools:

1. GetTopUsers(limit)  – returns the highest-ranked users.
2. sendMail(sender, receiver, subject, body)  – sends an email.

Workflow:
1. Find the top users.
2. Write a personalised thank-you email for each.
3. Send each email.
4. Repeat until every user has received one.
"""
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Reward our top 5 users.

Sender: lokeshvijayraina@gmail.com

Write a professional thank-you email for each user and send it.
"""
            )
        ]
    }
)


Success! Message ID: <202607051730.62862808934@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com itslokeshx@gmail.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.58777046012@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com ethan@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.24086269222@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com david@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.35837956438@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com liam@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051731.55185725366@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com sarah@example.com Thank You for Being a Top Listener
